# Long-term memory with a Store

- **Short-term memory** (checkpointer) = one conversation, keyed by `thread_id`.
- **Long-term memory** (store) = facts about a **user**, keyed by `user_id`, available in **every** thread.

```
START -> remember (LLM decides what's worth saving -> store) -> chat_node (reads memories from store) -> END
```

In [1]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import uuid

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b', reasoning_effort='low')

In [2]:
# First, the Store on its own: a key-value database organised by namespace (like folders)
store = InMemoryStore()

namespace = ('user-1', 'memories')                  # (user_id, kind of data)
store.put(namespace, 'fact-1', {'text': 'Likes cricket'})
store.put(namespace, 'fact-2', {'text': 'Lives in Jaipur'})

print('get one:   ', store.get(namespace, 'fact-1').value)
print('list all:  ', [item.value['text'] for item in store.search(namespace)])
print('other user:', store.search(('user-2', 'memories')))  # namespaces keep users apart

get one:    {'text': 'Likes cricket'}
list all:   ['Likes cricket', 'Lives in Jaipur']
other user: []


In [3]:
# The LLM decides what is worth remembering about the user
class MemoryDecision(BaseModel):
    should_save: bool = Field(description='True only if the message contains a lasting fact about the user')
    facts: list[str] = Field(description='Short facts about the user, e.g. "Name is Priyansh". Empty if nothing to save.')


# method='json_schema': Groq returns JSON that matches the schema directly.
# The default (a forced tool call) sometimes comes back empty with gpt-oss.
memory_extractor = model.with_structured_output(MemoryDecision, method='json_schema')

In [4]:
# Nodes can ask for `store` and `config`: LangGraph passes them in automatically
def remember(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    namespace = (config['configurable']['user_id'], 'memories')
    known = [item.value['text'] for item in store.search(namespace)]

    decision = memory_extractor.invoke(
        f'Already known about the user: {known}\n'
        f"New user message: {state['messages'][-1].content}\n"
        'Extract EVERY new lasting fact about the user (name, city, job, likes, goals), one fact per item, '
        'written as "User ..." (e.g. "User works as a data analyst"). Skip questions and small talk.'
    )
    if decision.should_save:
        for fact in decision.facts:
            store.put(namespace, str(uuid.uuid4()), {'text': fact})
    return {}  # this node only writes to the store, not to the conversation state


def chat_node(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    namespace = (config['configurable']['user_id'], 'memories')
    memories = '\n'.join(f"- {item.value['text']}" for item in store.search(namespace)) or '(nothing yet)'

    system = SystemMessage(f'You are a helpful assistant. What you know about this user:\n{memories}')
    return {'messages': [model.invoke([system] + state['messages'])]}

In [5]:
graph = StateGraph(MessagesState)
graph.add_node('remember', remember)
graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'remember')
graph.add_edge('remember', 'chat_node')
graph.add_edge('chat_node', END)

store = InMemoryStore()  # fresh, empty store for the chatbot
chatbot = graph.compile(checkpointer=InMemorySaver(), store=store)  # short-term AND long-term memory

In [6]:
def chat(text, user_id, thread_id):
    config = {'configurable': {'user_id': user_id, 'thread_id': thread_id}}
    result = chatbot.invoke({'messages': [HumanMessage(content=text)]}, config=config)
    print(f'[{user_id} / {thread_id}] You: {text}\nAI: {result["messages"][-1].content[:150]}\n')


# Conversation 1
chat('Hi! I am Priyansh from Jaipur. I work as a data analyst.', 'user-1', 'chat-1')
chat('I love cricket and I am learning LangGraph.', 'user-1', 'chat-1')

[user-1 / chat-1] You: Hi! I am Priyansh from Jaipur. I work as a data analyst.
AI: Hey Priyansh! 👋  
Nice to meet a data analyst from Jaipur. How can I help you today? Whether it’s crunching numbers, visualising data, or just brainst



[user-1 / chat-1] You: I love cricket and I am learning LangGraph.
AI: That’s awesome! 🎉

- **Cricket** – Jaipur’s a great city for cricket fans. Do you follow any particular team or player?  
- **LangGraph** – It’s a pow



In [7]:
# What got saved for user-1
for item in store.search(('user-1', 'memories')):
    print('-', item.value['text'])

- User name is Priyansh
- User is from Jaipur
- User works as a data analyst
- User likes cricket
- User is learning LangGraph


In [8]:
# Conversation 2: a NEW thread (empty short-term memory), same user -> still knows them
chat('Suggest a weekend plan for me.', 'user-1', 'chat-2')

# A different user in a new thread knows nothing about user-1
chat('What do you know about me?', 'user-2', 'chat-3')

[user-1 / chat-2] You: Suggest a weekend plan for me.
AI: Hey Priyansh!  
Here’s a weekend plan that mixes a bit of data fun, a cricket catch‑up, and a dash of Jaipur charm—so you get a relaxed yet productive



[user-2 / chat-3] You: What do you know about me?
AI: I don’t have any personal information about you. I only know what you share with me during our conversation. If you’d like to tell me a bit about your

